# 06 — XGBoost Optimisé : Visualisation & Performance
Ce notebook contient les graphiques avancés de performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

plt.style.use('ggplot')


In [ ]:
df = pd.read_csv('../features/ml_dataset.csv').dropna(subset=['result'])
features = ['home_elo', 'away_elo', 'elo_diff', 'home_form_5', 'away_form_5', 'home_avg_overall', 'away_avg_overall', 'odds_prob_home', 'odds_prob_draw', 'odds_prob_away']
X, y = df[features].fillna(0), LabelEncoder().fit_transform(df['result'])

X_train, y_train = X[df['season'] != '2024/25'], y[df['season'] != '2024/25']
X_test, y_test = X[df['season'] == '2024/25'], y[df['season'] == '2024/25']

model = xgb.XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.02, eval_metric='mlogloss')
model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test, y_test)], verbose=False)


## 1. Courbe d'Apprentissage (Loss Curve)
Indispensable pour vérifier que le modèle ne sur-apprend pas (overfitting).

In [ ]:
results = model.evals_result()
plt.figure(figsize=(10,6))
plt.plot(results['validation_0']['mlogloss'], label='Entraînement')
plt.plot(results['validation_1']['mlogloss'], label='Validation')
plt.title('Évolution de la Perte (LogLoss) par Itération')
plt.xlabel('Nombre d\'Arbres')
plt.ylabel('LogLoss')
plt.legend()
plt.show()

## 2. Matrice de Confusion (Heatmap)
Visualisation précise des erreurs par classe.

In [ ]:
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8,6))
sns.heatmap(cm_norm, annot=True, cmap='Blues', fmt='.2f', xticklabels=['Away', 'Draw', 'Home'], yticklabels=['Away', 'Draw', 'Home'])
plt.title('Matrice de Confusion Normalisée')
plt.xlabel('Prédiction')
plt.ylabel('Réalité')
plt.show()

## 3. Importance des Features
Quelles sont les variables qui dictent le score ?

In [ ]:
xgb.plot_importance(model, importance_type='gain', max_num_features=10, height=0.5)
plt.title('Importance des Features (Gain)')
plt.show()